```
Random Forest

Dataset Description:

Use the Glass dataset and apply the Random forest model.

1. Exploratory Data Analysis (EDA):

Perform exploratory data analysis to understand the structure of the dataset.
Check for missing values, outliers, inconsistencies in the data.

2: Data Visualization:

Create visualizations such as histograms, box plots, or pair plots to visualize the distributions and relationships between features.
Analyze any patterns or correlations observed in the data.

3: Data Preprocessing

1. Check for missing values in the dataset and decide on a strategy for handling them.Implement the chosen strategy (e.g., imputation or removal) and explain your reasoning.
2. If there are categorical variables, apply encoding techniques like one-hot encoding to convert them into numerical format.
3. Apply feature scaling techniques such as standardization or normalization to ensure that all features are on a similar scale. Handling the imbalance data.

4: Random Forest Model Implementation
1. Divide the data into train and test split.
2. Implement a Random Forest classifier using Python and a machine learning library like scikit-learn.
3. Train the model on the train dataset. Evaluate the performance on test data using metrics like accuracy, precision, recall, and F1-score.

5: Bagging and Boosting Methods
Apply the Bagging and Boosting methods and compare the results.


Additional Notes:
1. Explain Bagging and Boosting methods. How is it different from each other.
2. Explain how to handle imbalance in the data.
```

In [ ]:
#Understanding Data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np

xls_file = pd.ExcelFile('glass.xlsx')
print(xls_file.sheet_names)

dataset = pd.read_excel('glass.xlsx', sheet_name = 'glass')
print(dataset)

# Working on a copy
df = dataset.copy()

print("\n<----------INFO----------->\n")
print(df.info())

print("\n<-----------DESCRIBE ONLY NUMERICAL---------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL--------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES--------->\n")
print(df.isnull().sum())

## Exploratory Data Analysis(EDA):

In [ ]:
# Uivariate Analysis
target = 'Type'
numerical_cols = df.select_dtypes(include=['float64']).columns.tolist()
print(numerical_cols)

for col in numerical_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    plt.title(f"Histogram for {col}")
    sns.histplot(df[col],kde=True,bins=20)

    plt.subplot(1,2,2)
    plt.title(f"Boxplot of {col}")
    sns.boxplot(df[col])
    
    plt.show()

In [ ]:
# Bivariate Analysis

#Numerical variable vs target variable
for col in numerical_cols:
    plt.figure(figsize=(15,10))
    plt.title(f"{col} vs Type")
    sns.boxplot(x='Type', y=col,data=df)
    plt.xlabel('Type')
    plt.ylabel(col)

    plt.show()


In [ ]:
# Numerical variable vs Numerical variable

for i,x_col in enumerate(numerical_cols):
    for y_col in numerical_cols[i+1:]:
        plt.figure(figsize=(15,10))
        plt.title(f"{x_col} vs {y_col}")
        sns.scatterplot(x=x_col, y= y_col, data=df)

        plt.show()

## Data Cleaning 

In [ ]:
#Counting Outliers in different columns
outliers_summary ={}
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - IQR*1.5
    upper_limit = Q3 + IQR*1.5

    outliers = df[((df[col] < lower_limit)  |  (df[col] > upper_limit))][col]
    outliers_summary[col] = outliers.tolist()

outliers_value = pd.DataFrame(dict([ (k , pd.Series(v)) for k,v in outliers_summary.items()]))

print(outliers_value)


In [ ]:
# Cliping outliers 
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - IQR*1.5
    upper_limit = Q3 + IQR*1.5

    df[col] = np.clip(df[col],lower_limit, upper_limit)

In [ ]:
# Chekcing whethere the outliers has been removed or not
for col in numerical_cols:
    plt.figure(figsize=(15,10))
    sns.boxplot(df[col])
    plt.show()

## Data Partition and Model Building
#### Bagging:Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Dividing features into target and independent variables
X = df.iloc[:,:-1]
y= df.iloc[:,-1]

# Dividing dataset into train and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

#Implementing model
rf_model = RandomForestClassifier()

parameters = {
   'n_estimators': [100, 200],       
    'max_depth': [5, 10, None],       
    'min_samples_split': [2, 5], 
    'min_samples_leaf': [1, 2],
    'max_features': ["sqrt"]
}
rf_grid = GridSearchCV(rf_model,parameters,scoring="accuracy", cv=5)
rf_grid.fit(X_train, y_train)

In [ ]:
#Best parameters
print(rf_grid.best_params_)

print("\n<-------- Best CV Accuracy-------> \n")
print(rf_grid.best_score_)

In [ ]:
y_pred = rf_grid.predict(X_test)

# Test accuary acuracy
print("Test accuracy is given by :", accuracy_score(y_test,y_pred))

# Classification report
print(classification_report(y_test, y_pred))

#### Boosting: AdaBoost

In [ ]:
# implementing Adaboost and comapairingthere performance
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Base model 
base_estimator = DecisionTreeClassifier()

# AdaBoost model
ada_model = AdaBoostClassifier(base_estimator, random_state= 42)

parameters ={
    'n_estimators':[50,100,200],
    'learning_rate':[0.01,0.05,0.1,0.5,1]
}

ada_grid = GridSearchCV(ada_model,parameters,scoring = 'accuracy',cv=5)
ada_grid.fit(X_train,y_train)

In [ ]:
# Best parameters
print(ada_grid.best_params_)

print("\n<---------Best Cv score--------->\n")
print(ada_grid.best_score_)

In [ ]:
# Test Score and classification report
y_pred2 = ada_grid.predict(X_test)

print("\n<---------Test Score is given by ---------->\n")
print(accuracy_score(y_pred2,y_test))

print("\n<---------Classification Report--------->\n")
print(classification_report(y_pred2, y_test))

## Conclusion After Comapiring Both Techniques
```
Ans: From the above result, we can clearly see the accuracy score of the test for RandomForest(uses Bagging Technique),
is greater than the AdaBoost(uss Boosting Technique).
    Hence, The RandomForst model will perform well then the AdaBoost. 
```

## Notes:

#### Bagging Vs Boosting:
```
Bagging(Bootstrap Aggregating):
i. train multiple models known as weak learners (usually decision trees) independently on
different random subset of data.
ii.Take majority vote(classification problems), average(regression problems)
iii.sed to reduce variance and preventing overfitting.
 eg.RandomForest

Boosting:
i. Train models sequentially where each new model focuses on the errors of the previous one.
ii. Uses aweighted sum of all models prediction to predict the output.
iii. Reduces bias and improve accuracy.
    eg.AdaBoost,XGBoost, Gradient Boosting etc.
```

#### Handling Imbalanced Data
```
There are variaous way to handle the imbalanced  data :
1.) Resampling: Oversampling minority and undersampling the majority.
2.)Class weights: Assigning higher weights to the minority class.
3.)Algorithms: Using tree-ased models/ensemble methods.
4.) Metrics: using metrices like precision, Recall,F1-Score. 
```